# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides an end-to-end guide to loading, exploring, and analyzing the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided as a Croissant schema URL:
[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata and print overview
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n")
print(f"Identifier (DOI): {metadata.identifier}\n")
print(f"Description: {metadata.description}\n")
print(f"Temporal Coverage: {metadata.temporalCoverage}\n")
print(f"Spatial Coverage: {metadata.spatialCoverage}\n")
print(f"Keywords: {', '.join(metadata.keywords) if hasattr(metadata, 'keywords') else 'N/A'}\n")
print(f"License: {metadata.license}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

**Note:** All Croissant entities (record sets, fields, columns) are referenced by their `@id` for consistency with schema semantics.

In [ ]:
# List all record sets and their IDs
record_sets = list(dataset.record_sets)
if len(record_sets) == 0:
    print("No record sets found in metadata.\n")
else:
    print("Available Record Sets:")
    for rs in record_sets:
        print(f"- @id: {rs['@id']}, name: {rs.get('name','(no name)')}")

# For demonstration, try the first record set (if one exists)
if len(record_sets) > 0:
    record_set_id = record_sets[0]['@id']
    print(f"\nFields for Record Set '{record_set_id}':")
    fields = dataset.fields(record_set=record_set_id)
    for field in fields:
        print(f"  - @id: {field['@id']} | name: {field.get('name',(field.get('@id','')))})")

# Show a sample record (first record set only)
if len(record_sets) > 0:
    print(f"\nSample record from record set '{record_set_id}':")
    record_generator = dataset.records(record_set=record_set_id)
    try:
        for idx, rec in zip(range(1), record_generator):
            print(rec)
    except Exception as e:
        print(f"Error retrieving records: {e}")
else:
    print("No record sets available to display a sample record.")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for further analysis.

All extraction and referencing uses Croissant `@id` fields.

In [ ]:
# Build DataFrames for each record set using their @id
dataframes = {}
if len(record_sets) == 0:
    print("No record sets to extract data from.")
else:
    print("Extracting data for record sets:")
    for rs in record_sets:
        rs_id = rs['@id']
        print(f"- {rs_id}")
        try:
            records = list(dataset.records(record_set=rs_id))
            if records:
                dataframes[rs_id] = pd.DataFrame(records)
            else:
                print(f"  No records found for {rs_id}.")
        except Exception as e:
            print(f"  Error loading {rs_id}: {e}")
    # If any record sets loaded, display their column names and first few rows
    if len(dataframes) > 0:
        # Select first available DataFrame
        sample_rs = list(dataframes.keys())[0]
        print(f"\nColumns in '{sample_rs}':\n", dataframes[sample_rs].columns.tolist())
        display(dataframes[sample_rs].head())
    else:
        print("No dataframes loaded for analysis.")

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate some common EDA steps using fields referenced by their `@id`.

- Select a numeric column by its Croissant `@id` and filter records.
- Normalize values, show summary statistics.
- Optionally group by a categorical field `@id` if available.

In [ ]:
# ---- PARAMETER: set appropriate IDs below (update based on your dataset schema) ----
# If no data loaded, skip. Otherwise, use first available DataFrame/record set.
if len(dataframes) == 0:
    print("No record sets loaded, cannot perform EDA.")
else:
    # Pick the first loaded DataFrame for demonstration
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Try to pick a numeric field automatically, otherwise prompt the user
    numeric_col_candidates = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    if len(numeric_col_candidates) == 0:
        print("No numeric columns found in record set for EDA.")
    else:
        numeric_field_id = numeric_col_candidates[0]  # Use first numeric field as example
        threshold = df[numeric_field_id].mean()  # Use mean as a threshold demonstration
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records in '{record_set_id}' with {numeric_field_id} > {threshold:.3f}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}':")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a likely categorical field (string or object dtype, not id column)
        group_cols = df.select_dtypes(include=['object','category']).columns
        group_field = None
        for col in group_cols:
            # Avoid grouping by id-like fields
            if col != numeric_field_id and not col.lower().endswith(('id', '@id')):
                group_field = col
                break
        if group_field:
            print(f"\nGrouped by '{group_field}':")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
            display(grouped_df.head())
        else:
            print("No suitable categorical field for grouping in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Proceed only if we have a DataFrame and a numeric column
if len(dataframes) == 0 or ('numeric_field_id' not in locals()):
    print("No numeric field or DataFrame available for visualization.")
else:
    # Distribution plot
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in Record Set '{record_set_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    
    # If at least two numeric fields, do a scatter plot
    if len(df.select_dtypes(include=['float64', 'int64']).columns) > 1:
        x_col = numeric_field_id
        y_col = df.select_dtypes(include=['float64', 'int64']).columns[1]
        plt.figure(figsize=(7,7))
        plt.scatter(df[x_col], df[y_col], alpha=0.7)
        plt.xlabel(x_col)
        plt.ylabel(y_col)
        plt.title(f"Scatter: {x_col} vs. {y_col}")
        plt.grid(True)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load a FAIR^2 Croissant-annotated dataset using `mlcroissant`, explored its metadata and record structure (referencing all entities by their Croissant `@id`), extracted and transformed tabular data, and visualized key numerical fields. This process provides a reproducible foundation for transparent, FAIR-aligned data analysis workflows.

**Next Steps:** Deeper domain analysis, linking with other Croissant datasets, advanced visualization, and model building.